In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import math

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X = X[:, :500] 
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(-1)
X_val   = X_val.unsqueeze(-1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0) #for gpu

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(1, 64)

        self.pos_encoder = PositionalEncoding(64)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.embedding(x)

        x = self.pos_encoder(x)

        x = self.transformer(x)

        x = x.mean(dim=1)   # global pooling - instead of last timestamp this uses whole sequence 

        x = self.fc(x)

        return x

model = TransformerModel()


class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))  ##using only 500 sequence length instead of 3000 and reduced bathc size form 32 to 16

Epoch 1, Train Loss: 175.8719
Epoch 2, Train Loss: 174.9735
Epoch 3, Train Loss: 176.0587
Epoch 4, Train Loss: 176.4640
Epoch 5, Train Loss: 176.6775
Epoch 6, Train Loss: 176.7109
Epoch 7, Train Loss: 176.7472
Epoch 8, Train Loss: 176.6829
Epoch 9, Train Loss: 176.7293
Epoch 10, Train Loss: 176.6543
Epoch 11, Train Loss: 176.7563
Epoch 12, Train Loss: 176.7485
Epoch 13, Train Loss: 176.7487
Epoch 14, Train Loss: 176.6764
Epoch 15, Train Loss: 176.5524
Epoch 16, Train Loss: 176.4974
Epoch 17, Train Loss: 176.7623
Epoch 18, Train Loss: 176.7537
Epoch 19, Train Loss: 176.7073
Epoch 20, Train Loss: 176.7710
Total evaluated samples: 1018
Validation Accuracy: 0.0068762279115617275
[[   0 1011]
 [   0    7]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1011
           1       0.01      1.00      0.01         7

    accuracy                           0.01      1018
   macro avg       0.00      0.50      0.01      1018
weighted avg     

e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import math

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X = X[:, :300] 
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(-1)
X_val   = X_val.unsqueeze(-1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0) #for gpu

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(1, 64)

        self.pos_encoder = PositionalEncoding(64)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.embedding(x)

        x = self.pos_encoder(x)

        x = self.transformer(x)

        x = x.mean(dim=1)   # global pooling - instead of last timestamp this uses whole sequence 

        x = self.fc(x)

        return x

model = TransformerModel()

class_counts = torch.bincount(y_train)
weights = 1.0 / class_counts.float()
weights = weights/weights.sum()

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))  ##removed sampler + reduced sequence from 500 to 300

Epoch 1, Train Loss: 15.5148
Epoch 2, Train Loss: 12.3889
Epoch 3, Train Loss: 11.8015
Epoch 4, Train Loss: 12.0314
Epoch 5, Train Loss: 12.0734
Epoch 6, Train Loss: 13.1465
Epoch 7, Train Loss: 12.2565
Epoch 8, Train Loss: 12.1039
Epoch 9, Train Loss: 12.0473
Epoch 10, Train Loss: 12.5712
Epoch 11, Train Loss: 11.8327
Epoch 12, Train Loss: 12.4120
Epoch 13, Train Loss: 12.3713
Epoch 14, Train Loss: 12.6911
Epoch 15, Train Loss: 11.9993
Epoch 16, Train Loss: 11.3693
Epoch 17, Train Loss: 12.2360
Epoch 18, Train Loss: 12.1668
Epoch 19, Train Loss: 11.7444
Epoch 20, Train Loss: 11.9682
Total evaluated samples: 1018
Validation Accuracy: 0.9931237697601318
[[1011    0]
 [   7    0]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1011
           1       0.00      0.00      0.00         7

    accuracy                           0.99      1018
   macro avg       0.50      0.50      0.50      1018
weighted avg       0.99      0.99      0

e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Programming_vsc\exoplanets\detecting exoplanets raw photometric\dlenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio